## 1. Installation des dépendances

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'selenium', 'webdriver-manager', '-q'])

0

## 2. Imports

In [2]:
import json
import csv
import time
import re
import os

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

## 3. Configuration

In [4]:
SEARCH_URLS = {
    'vente':    'https://www.orpi.com/recherche/buy/',
    'location': 'https://www.orpi.com/recherche/rent/',
}

CARD_SELECTOR     = 'c-results__list__item'
ANNONCE_URL_REGEX = re.compile(r'https://www\.orpi\.com/annonce-[^/?#]+/')

MAX_PAGES    = 20     
DELAY_PAGE   = 3.0 
DELAY_DETAIL = 2.0   
OUTPUT_DIR   = '../data'

os.makedirs(OUTPUT_DIR, exist_ok=True)
estimation = MAX_PAGES * 14 * 2

## 4. Initialisation du navigateur

In [5]:
def build_driver():
    
    opts = Options()
    opts.add_argument('--headless')
    opts.add_argument('--no-sandbox')
    opts.add_argument('--disable-dev-shm-usage')
    opts.add_argument('--window-size=1920,1080')
    opts.add_argument(
        'user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
    )
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)

def dismiss_cookies(driver):

    try:
        btn = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH,
                "//button[contains(text(),'Continuer sans') "
                "or contains(text(),'Refuser') "
                "or contains(text(),'Tout refuser')]"
            ))
        )
        btn.click()
        time.sleep(1)
    except Exception:
        pass

driver = build_driver()
driver.get('https://www.orpi.com')
time.sleep(2)

## 5. Fonctions de parsing

In [6]:
def parse_prix(text: str):
    m = re.search(r'([\d\s]+)\s*€', text.replace('\xa0', ' '))
    if m:
        try:
            return float(m.group(1).replace(' ', ''))
        except ValueError:
            pass
    return None

def parse_surface(text: str):
    m = re.search(r'(\d+[\.,]?\d*)\s*m2', text, re.IGNORECASE)
    if m:
        try:
            return float(m.group(1).replace(',', '.'))
        except ValueError:
            pass
    return None

def parse_nb_pieces(text: str):
    m = re.search(r'(\d+)\s*pi[eè]ces?', text, re.IGNORECASE)
    return int(m.group(1)) if m else None

def parse_type_bien(text: str):
    for t in ['Maison', 'Appartement', 'Studio', 'Terrain',
              'Villa', 'Immeuble', 'Local', 'Bureau', 'Parking']:
        if t.lower() in text.lower():
            return t
    return ''

def parse_ville(text: str):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    for i, line in enumerate(lines):
        if re.search(r'\d+\s*m2', line, re.IGNORECASE):
            if i + 1 < len(lines):
                ville_line = lines[i + 1]
                for noise in ['Message', 'Favoris', 'chambres', 'chambre']:
                    ville_line = ville_line.replace(noise, '').strip()
                ville = ville_line.split('-')[0].strip()
                if ville:
                    return ville
    return ''

def parse_card(card_element, type_transaction: str):

    text = card_element.text.strip()
    if not text:
        return None

    prix    = parse_prix(text)
    surface = parse_surface(text)
    if prix is None or surface is None:
        return None

    url = ''
    for link in card_element.find_elements(By.TAG_NAME, 'a'):
        href = link.get_attribute('href') or ''
        if ANNONCE_URL_REGEX.match(href) and 'contact' not in href:
            url = href
            break

    return {
        'type_transaction': type_transaction,
        'type_bien':        parse_type_bien(text),
        'prix':             prix,
        'surface':          surface,
        'nb_pieces':        parse_nb_pieces(text),
        'ville':            parse_ville(text),
        'code_postal':      '',
        'description':      '',
        'source':           'orpi',
        '_url':             url,
    }

## 6. Phase 1 — Collecte des cartes d'annonces

In [6]:
all_annonces = []
cookies_dismissed = False

for transaction, base_url in SEARCH_URLS.items():
    print(f"\n{'='*55}")
    print(f"  Type : {transaction.upper()} — {MAX_PAGES} pages max")
    print(f"{'='*55}")

    for page in range(1, MAX_PAGES + 1):
        url = f'{base_url}?page={page}'
        print(f'  Page {page}/{MAX_PAGES}...', end=' ', flush=True)

        driver.get(url)
        time.sleep(DELAY_PAGE)

        # Ferme le bandeau cookies une seule fois
        if not cookies_dismissed:
            dismiss_cookies(driver)
            cookies_dismissed = True

        cards = driver.find_elements(By.CLASS_NAME, CARD_SELECTOR)
        parsed = [parse_card(c, transaction) for c in cards]
        valid  = [a for a in parsed if a is not None]

        if not valid:
            print('aucune annonce — fin de pagination')
            break

        all_annonces.extend(valid)
        print(f'{len(valid)} annonces  (cumul : {len(all_annonces)})')

print(f"\n✅ Phase 1 terminée — {len(all_annonces)} cartes collectées")


  Type : VENTE — 20 pages max
  Page 1/20... 13 annonces  (cumul : 13)
  Page 2/20... 14 annonces  (cumul : 27)
  Page 3/20... 13 annonces  (cumul : 40)
  Page 4/20... 13 annonces  (cumul : 53)
  Page 5/20... 14 annonces  (cumul : 67)
  Page 6/20... 14 annonces  (cumul : 81)
  Page 7/20... 14 annonces  (cumul : 95)
  Page 8/20... 14 annonces  (cumul : 109)
  Page 9/20... 13 annonces  (cumul : 122)
  Page 10/20... 14 annonces  (cumul : 136)
  Page 11/20... 14 annonces  (cumul : 150)
  Page 12/20... 14 annonces  (cumul : 164)
  Page 13/20... 13 annonces  (cumul : 177)
  Page 14/20... 14 annonces  (cumul : 191)
  Page 15/20... 12 annonces  (cumul : 203)
  Page 16/20... 11 annonces  (cumul : 214)
  Page 17/20... 13 annonces  (cumul : 227)
  Page 18/20... 14 annonces  (cumul : 241)
  Page 19/20... 14 annonces  (cumul : 255)
  Page 20/20... 14 annonces  (cumul : 269)

  Type : LOCATION — 20 pages max
  Page 1/20... 14 annonces  (cumul : 283)
  Page 2/20... 14 annonces  (cumul : 297)
  Page 

## 7. Phase 2 — Extraction des descriptions

In [7]:
# Mots-clés parasites à exclure (cookies, CGU, formulaires…)
EXCLUSIONS = {
    'honoraires', 'georisques', 'bloctel', 'cookies', 'gagner du temps',
    'newsletter', 'inscription', 'recevoir', 'navigation', 'informations',
    'visite', 'personnalis', 'consentement', 'partenaires', 'publicit',
    'abonnez', 'contact', 'téléphone', 'offre commerciale',
    'conseil', 'chiffres clés', 'opportunité',
}

# Mots-clés typiques d'une vraie description immobilière
IMMO_KEYWORDS = {
    'pièce', 'chambre', 'cuisine', 'séjour', 'salle', 'terrasse',
    'jardin', 'étage', 'lumineux', 'rénov', 'parking', 'cave',
    'm²', 'm2', 'garage', 'quartier', 'proche', 'commerces',
    'belle', 'beau', 'spacieux', 'calme', 'vue', 'balcon',
}

def extract_detail(url: str):

    result = {'description': '', 'code_postal': ''}
    if not url:
        return result

    try:
        driver.get(url)
        time.sleep(DELAY_DETAIL)

        # Tente d'abord les sélecteurs CSS ciblés
        desc_selectors = [
            "[class*='description']",
            "[class*='c-estate-description']",
            "[class*='estate-text']",
        ]
        for sel in desc_selectors:
            try:
                el = driver.find_element(By.CSS_SELECTOR, sel)
                txt = el.text.strip()
                if len(txt) > 60:
                    result['description'] = txt
                    break
            except Exception:
                pass

        # Fallback : cherche un paragraphe long avec du vocabulaire immobilier
        if not result['description']:
            body_text = driver.find_element(By.TAG_NAME, 'body').text
            lines = [l.strip() for l in body_text.split('\n') if len(l.strip()) > 80]
            for line in lines:
                line_lower = line.lower()
                if any(excl in line_lower for excl in EXCLUSIONS):
                    continue
                if any(kw in line_lower for kw in IMMO_KEYWORDS):
                    result['description'] = line
                    break

        m = re.search(r'-(\d{5})-', url)
        if m:
            result['code_postal'] = m.group(1)

    except Exception as e:
        print(f'    [ERREUR] {e}')

    return result


In [8]:
total = len(all_annonces)
print(f'Phase 2 — visite de {total} fiches individuelles...')
print(f'Durée estimée : ~{total * DELAY_DETAIL / 60:.0f} min\n')

for i, annonce in enumerate(all_annonces):
    url = annonce.get('_url', '')
    print(f'  [{i+1:>3}/{total}] {url[:65]}', end=' ', flush=True)

    detail = extract_detail(url)
    annonce['description'] = detail['description']
    annonce['code_postal'] = detail['code_postal'] or annonce['code_postal']

    has_desc = '✓' if annonce['description'] else '✗'
    print(f'desc={has_desc}')

# Fermeture du driver
driver.quit()
print(f'\n✅ Phase 2 terminée — driver fermé')

# Suppression de la clé interne _url
for a in all_annonces:
    a.pop('_url', None)

with_desc = sum(1 for a in all_annonces if a.get('description', '').strip())
print(f'   Annonces avec description : {with_desc}/{total}')

Phase 2 — visite de 544 fiches individuelles...
Durée estimée : ~18 min

  [  1/544] https://www.orpi.com/annonce-vente-maison-t5-ay-51160-5b131461-7c desc=✓
  [  2/544] https://www.orpi.com/annonce-vente-maison-t4-toulouse-31200-7252b desc=✓
  [  3/544] https://www.orpi.com/annonce-vente-maison-t3-toulouse-31200-8c756 desc=✓
  [  4/544] https://www.orpi.com/annonce-vente-maison-t9-villematier-31340-6b desc=✓
  [  5/544] https://www.orpi.com/annonce-vente-maison-t10-villematier-31340-d desc=✓
  [  6/544] https://www.orpi.com/annonce-vente-appartement-t3-aucamville-3114 desc=✓
  [  7/544] https://www.orpi.com/annonce-vente-maison-t5-saint-alban-31140-0d desc=✓
  [  8/544] https://www.orpi.com/annonce-vente-appartement-t4-fontenay-aux-ro desc=✓
  [  9/544] https://www.orpi.com/annonce-vente-appartement-t3-toulouse-31200- desc=✓
  [ 10/544] https://www.orpi.com/annonce-vente-appartement-t4-charenton-le-po desc=✓
  [ 11/544] https://www.orpi.com/annonce-vente-maison-t3-vitry-en-artois-6249

## 8. Sauvegarde des données

In [9]:
# --- JSON ---
json_path = os.path.join(OUTPUT_DIR, 'orpi_raw.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(all_annonces, f, ensure_ascii=False, indent=2)
print(f'JSON sauvegardé → {json_path}')

# --- CSV ---
csv_path = os.path.join(OUTPUT_DIR, 'orpi_raw.csv')
if all_annonces:
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=all_annonces[0].keys())
        writer.writeheader()
        writer.writerows(all_annonces)
    print(f'CSV sauvegardé  → {csv_path}')

print(f'\nTotal : {len(all_annonces)} annonces exportées')

JSON sauvegardé → ../data/orpi_raw.json
CSV sauvegardé  → ../data/orpi_raw.csv

Total : 544 annonces exportées
